In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = r"C:\git\CleverCheck\server\my_model\hebert_model_download"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)

model.eval()  # קריטי לבדיקות

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    # אם זה regression (ציון רציף)
    score = logits.item()

    return score


# -------------------------
# בדיקות
# -------------------------
samples = [
    "התשובה נכונה ומדויקת מאוד",
    "התשובה חלקית ולא ברורה",
    "אין קשר לשאלה בכלל"
]

for s in samples:
    print(s)
    print("Score:", predict(s))
    print("-" * 40)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8278.07it/s]

התשובה נכונה ומדויקת מאוד


Score: 0.8019910454750061
----------------------------------------
התשובה חלקית ולא ברורה
Score: 0.39667898416519165
----------------------------------------
אין קשר לשאלה בכלל
Score: 0.16000625491142273
----------------------------------------


In [ ]:


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()  # חובה

# -------------------------
# פונקציית בניית קלט
# -------------------------
def build_input(question, reference, student):
    return f"שאלה: {question} תשובת מורה: {reference} תשובת תלמיד: {student}"


# -------------------------
# חיזוי
# -------------------------
def predict(question, reference, student):
    text = build_input(question, reference, student)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # regression score
    return outputs.logits.item()


# -------------------------
# בדיקות לדוגמה
# -------------------------
tests = [
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "סיבוב כדור הארץ סביב צירו.",
        "expected": "HIGH"
    },
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש זזה וגורמת ליום ולילה",
        "expected": "LOW"
    },
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "כדור הארץ מסתובב והשמש לא זזה",
        "expected": "MEDIUM"
    }
]

print("START EVALUATION\n")

for t in tests:
    score = predict(t["question"], t["reference"], t["student"])

    print("Q:", t["question"])
    print("Student:", t["student"])
    print("Score:", round(score, 3))
    print("Expected:", t["expected"])
    print("-" * 50)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()


def build_input(q, ref, student):
    return f"שאלה: {q} תשובת מורה: {ref} תשובת תלמיד: {student}"


def predict(q, ref, student):
    text = build_input(q, ref, student)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        out = model(**inputs)

    return out.logits.item()


# -------------------------
# סט בדיקות לפי אורכים
# -------------------------
tests = [

    # 🟢 קצר ונכון
    {
        "type": "SHORT_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "כדור הארץ מסתובב סביב עצמו."
    },

    # 🟢 קצר שגוי
    {
        "type": "SHORT_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש מסתובבת סביב כדור הארץ."
    },

    # 🟡 בינוני נכון
    {
        "type": "MEDIUM_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "יום ולילה נוצרים בגלל סיבוב כדור הארץ סביב צירו ביחס לשמש."
    },

    # 🟡 בינוני שגוי
    {
        "type": "MEDIUM_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש זזה ומסתובבת סביב כדור הארץ וגורמת ליום ולילה."
    },

    # 🔵 ארוך נכון
    {
        "type": "LONG_CORRECT",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "תופעת היום והלילה נגרמת כתוצאה מסיבוב כדור הארץ סביב צירו, כאשר חלקים שונים של הכוכב פונים לשמש וחווים אור, בעוד החלקים האחרים נמצאים בחושך."
    },

    # 🔵 ארוך שגוי
    {
        "type": "LONG_WRONG",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "השמש היא זו שמסתובבת סביב כדור הארץ בתנועה יומית ולכן נוצר יום ולילה, כאשר היא נעה מצד אחד לשני של השמים."
    },

    # ⚫ רעש + ארוך
    {
        "type": "LONG_NOISY",
        "q": "מה גורם ליום ולילה?",
        "ref": "סיבוב כדור הארץ סביב צירו.",
        "student": "סיבוב כדור הארץ!!!! סביב הציר שלו??? גורם ליום ולילה בצורה מחזורית מאוד מעניינת!!! כאשר יש אור וחושך לסירוגין..."
    }
]


# -------------------------
# הרצה וסטטיסטיקה לפי סוג
# -------------------------
results = {}

print("START LENGTH EVALUATION\n")

for t in tests:
    score = predict(t["q"], t["ref"], t["student"])

    print(t["type"])
    print("SCORE:", round(score, 4))
    print("-" * 50)

    results.setdefault(t["type"], []).append(score)


print("\nSUMMARY\n")
for k, v in results.items():
    print(k, "AVG:", round(np.mean(v), 4))

In [ ]:
import pandas as pd
import re

file_path = r'C:\Users\levm\Desktop\train.csv'

df = pd.read_csv(file_path, encoding="utf-8-sig")

scores = []


import re

def parse(text):
    q_match = re.search(r"\[Q\](.*?)\[T\]", text)
    t_match = re.search(r"\[T\](.*?)\[S\]", text)
    s_match = re.search(r"\[S\](.*)", text)  # בלי פסיק!

    if not q_match or not t_match or not s_match:
        print("SKIP LINE (bad format):", text)
        return None

    q = q_match.group(1).strip()
    ref = t_match.group(1).strip()
    student = s_match.group(1).strip()

    return q, ref, student
for item in df["text"]:
    q, ref, student = parse(item)
    pred = predict(q, ref, student)
    scores.append(pred)

print("MIN:", min(scores))
print("MAX:", max(scores))
print("AVG:", sum(scores)/len(scores))

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import re

model_path = r"C:\git\CleverCheck\server\my_model\hebert_model_download"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("num_labels =", model.config.num_labels)
print("problem_type =", model.config.problem_type)

def parse_row(text):
    q = re.search(r"\[Q\](.*?)\[T\]", text)
    t = re.search(r"\[T\](.*?)\[S\]", text)
    s = re.search(r"\[S\](.*)", text)

    return (
        q.group(1).strip() if q else "",
        t.group(1).strip() if t else "",
        s.group(1).strip() if s else ""
    )


def build_input(q, t, s):
    return f"שאלה: {q}\nתשובת מורה: {t}\nתשובת תלמיד: {s}"


def predict_batch(batch):
    texts = [build_input(q, t, s) for q, t, s in batch]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
       outputs = model(**inputs)

    return outputs.logits.cpu().numpy().flatten()


# ---------------------------
# LOAD CSV
# ---------------------------
file_path = 'C:\\git\\CleverCheck\\server\\services\\train.csv'


df = pd.read_csv(file_path, encoding="utf-8-sig")


y_true = df.iloc[:, 1].values  # הציון (עמודה שנייה)
texts = df.iloc[:, 0].values   # הטקסט המשולב

y_pred = []

batch_size = 32

for i in range(0, len(df), batch_size):
    chunk = texts[i:i+batch_size]

    batch = [parse_row(t) for t in chunk]

    preds = predict_batch(batch)
    y_pred.extend(preds)


y_true = np.array(y_true, dtype=float)
y_pred = np.array(y_pred)


# ---------------------------
# METRICS
# ---------------------------
mae = np.mean(np.abs(y_true - y_pred))
corr = np.corrcoef(y_true, y_pred)[0, 1]
acc = np.mean(np.abs(y_true - y_pred) < 0.2)

print("MAE:", mae)
print("Correlation:", corr)
print("Threshold accuracy:", acc)


# ---------------------------
# PLOTS
# ---------------------------
plt.figure()
plt.hist(y_pred, bins=30)
plt.title("Predicted Distribution")
plt.show()

plt.figure()
plt.scatter(y_true, y_pred, s=5)
plt.title("True vs Predicted")
plt.show()
print(y_pred.min())
print(y_pred.max())
print(y_pred.mean())

In [ ]:
print(y_true.min())
print(y_true.max())

print(np.percentile(y_true, 5))
print(np.percentile(y_true, 95))
print(y_pred.min())
print(y_pred.max())
print(y_pred.mean())

In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = r"C:\git\CleverCheck\server\my_model\hebert_model_download"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()


def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # יציב גם לעתיד וגם ל-GPU
    return outputs.logits.cpu().numpy().flatten()[0]


# ---------------------------
# LOAD DATA
# ---------------------------
file_path = r"C:\git\CleverCheck\server\services\train.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig")

# בטיחות מול floatים
df.iloc[:, 1] = df.iloc[:, 1].round(2)

df_0 = df[df.iloc[:, 1] == 0.0].head(20)
df_1 = df[df.iloc[:, 1] == 1.0].head(20)


print("\n===== SCORE = 0.0 =====\n")

for _, row in df_0.iterrows():
    text = row.iloc[0]
    true_score = row.iloc[1]

    pred = predict(text)

    print("TRUE:", true_score)
    print("PRED:", round(pred, 4))
    print("TEXT:", text[:80])
    print("-" * 40)


print("\n===== SCORE = 1.0 =====\n")

for _, row in df_1.iterrows():
    text = row.iloc[0]
    true_score = row.iloc[1]

    pred = predict(text)

    print("TRUE:", true_score)
    print("PRED:", round(pred, 4))
    print("TEXT:", text[:80])
    print("-" * 40)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7503.03it/s]



===== SCORE = 0.0 =====

TRUE: 0.0
PRED: 0.3761
TEXT: [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיום ב-1918, פירוק האימפריה האוסטרו-ה
----------------------------------------
TRUE: 0.0
PRED: 0.9624
TEXT: [Q] איפה זורחת השמש ואיפה שוקעת? [T] השמש זורחת במזרח ושוקעת במערב. [S] השמש זור
----------------------------------------
TRUE: 0.0
PRED: 0.3666
TEXT: [Q] איפה זורחת השמש ואיפה שוקעת? [T] השמש זורחת במזרח ושוקעת במערב. [S] השמש עול
----------------------------------------
TRUE: 0.0
PRED: -0.0147
TEXT: [Q] איפה זורחת השמש ואיפה שוקעת? [T] השמש זורחת במזרח ושוקעת במערב. [S] לא יודע.
----------------------------------------
TRUE: 0.0
PRED: 0.9674
TEXT: [Q] איפה זורחת השמש ואיפה שוקעת? [T] השמש זורחת במזרח ושוקעת במערב. [S] השמש זור
----------------------------------------
TRUE: 0.0
PRED: 0.2794
TEXT: [Q] מה גורם ליום ולילה? [T] סיבוב כדור הארץ סביב צירו. [S] השמש מסתובבת סביב כדו
----------------------------------------
TRUE: 0.0
PRED: 0.058
TEXT: [Q] מה גורם ליום ולילה? [T] סיבוב כדור 

In [17]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# 1. הגדרות
# =========================
MODEL_PATH = r"C:\git\CleverCheck\server\my_model\hebert_model_download"
INPUT_CSV = r"C:\git\CleverCheck\server\services\test.csv"
OUTPUT_CSV = r"C:\git\CleverCheck\server\services\predictions.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 2. מודל
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()

# =========================
# 3. בניית input (חייב זהה לאימון!)
# =========================

# =========================
# 4. חיזוי
# =========================
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    return out.logits.squeeze().item()

# =========================
# 5. הרצה על CSV
# =========================
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

scores = []

for idx, row in df.iterrows():
    score = predict(row.iloc[0])

    scores.append(score)
    print(row.iloc[0])
    print(f"score {row.iloc[1]} → SCORE: {score:.4f}")

# =========================
# 6. שמירה
# =========================
df["predicted_score"] = scores
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nDONE ✔ קובץ נשמר עם predicted_score")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8926.04it/s]


[Q] מה גורם ליום ולילה? [T] סיבוב כדור הארץ סביב עצמו [S] כדור הארץ מסתובב סביב עצמו
score 1.0 → SCORE: 0.9435
[Q] מה גורם ליום ולילה? [T] סיבוב כדור הארץ סביב עצמו [S] השמש זזה
score 0.2 → SCORE: 0.1087
[Q] מהי מערכת השמש? [T] השמש וכוכבי הלכת שמקיפים אותה [S] השמש וכוכבים
score 0.6 → SCORE: 0.5683
[Q] מהי מערכת השמש? [T] השמש וכוכבי הלכת שמקיפים אותה [S] השמש וכוכבי הלכת
score 1.0 → SCORE: 0.6246
[Q] מהי פוטוסינתזה? [T] תהליך שבו צמחים משתמשים באור שמש ליצירת אנרגיה [S] צמחים משתמשים באור כדי לגדול
score 0.8 → SCORE: 0.4224
[Q] מהי פוטוסינתזה? [T] תהליך שבו צמחים משתמשים באור שמש ליצירת אנרגיה [S] תהליך בצמחים
score 0.4 → SCORE: 0.2333
[Q] מי היה דוד בן גוריון? [T] ראש הממשלה הראשון של ישראל [S] ראש הממשלה הראשון של ישראל
score 1.0 → SCORE: 0.8025
[Q] מי היה דוד בן גוריון? [T] ראש הממשלה הראשון של ישראל [S] מנהיג ישראל
score 0.7 → SCORE: 0.7162
[Q] מהו נהר? [T] זרם מים טבעי [S] מים זורמים בטבע
score 1.0 → SCORE: 0.3331
[Q] מהו נהר? [T] זרם מים טבעי [S] גוף מים גדול
score 0.5 → SCORE:

In [ ]:
import pandas as pd
import re
import random

INPUT_FILE = r"C:\git\CleverCheck\server\services\test.csv"
OUTPUT_FILE = r"C:\git\CleverCheck\server\services\augmented_2000.csv"

df = pd.read_csv(INPUT_FILE, header=None, encoding="utf-8-sig")

augmented = []
seen_q = set()

# -------------------------
# parser יציב
# -------------------------
def safe_parse(line):
    try:
        if "," not in line:
            return None

        text, score = line.rsplit(",", 1)

        q_start = text.find("[Q]")
        t_start = text.find("[T]")
        s_start = text.find("[S]")

        if q_start == -1 or t_start == -1 or s_start == -1:
            return None

        q = text[q_start+3:t_start].strip()
        ref = text[t_start+3:s_start].strip()
        student = text[s_start+3:].strip()

        return q, ref, student

    except:
        return None


# -------------------------
# גיוון אמיתי (חשוב!)
# -------------------------
def weak_answer(q):
    return random.choice([
        "חלק מהמידע לא מדויק",
        "תשובה כללית מדי",
        "חסר פירוט בתשובה",
        "יש רק הבנה חלקית",
        "התשובה לא מלאה"
    ])

def shorten(text):
    words = text.split()
    return " ".join(words[:max(2, len(words)//3)])

def drop_info(text):
    words = text.split()
    if len(words) <= 3:
        return "תשובה חלקית"
    return " ".join(words[:-4])


# -------------------------
# יצירה
# -------------------------
TARGET = 2000

for line in df[0]:

    if len(augmented) >= TARGET:
        break

    parsed = safe_parse(line)
    if not parsed:
        continue

    q, ref, student = parsed

    # 🚨 מניעת חזרות
    if q in seen_q:
        continue
    seen_q.add(q)

    # 🟥 1. קיצור תשובה
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {shorten(student)}",
        0.0
    ])

    if len(augmented) >= TARGET:
        break

    # 🟧 2. הורדת מידע
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {drop_info(student)}",
        0.0
    ])

    if len(augmented) >= TARGET:
        break

    # 🟨 3. תשובה כללית אמיתית (לא חזרה)
    augmented.append([
        f"[Q] {q} [T] {ref} [S] {weak_answer(q)}",
        0.0
    ])


# -------------------------
# חיתוך ושמירה
# -------------------------
augmented = augmented[:TARGET]

df_out = pd.DataFrame(augmented)
df_out.to_csv(OUTPUT_FILE, index=False, header=False, encoding="utf-8-sig")

print("DONE ✔ 2000 דוגמאות מגוונות נוצרו בלי חזרות")